In [6]:
import numpy as np

In [7]:
# --- 1. Load your simulation history ---
checkpoint_file = "simulation_checkpoint.npz"
data = np.load(checkpoint_file)

history_steps = data['history_steps']
history_energy = data['history_energy']


In [ ]:
# --- 2. Isolate the Equilibrated Region ---
# Discard the initial minimization "burn-in" (e.g., first 20000 steps)
burn_in = 20000  
eq_energy = history_energy[burn_in:]
eq_steps = history_steps[burn_in:]

# 3. Print the lengths to see what we have left
print(f"Total steps in file: {len(history_steps)}")
print(f"Steps left after removing burn-in: {len(eq_steps)}")

In [10]:
# 1. Calculate the average (mean) energy of your stable data
mean_E = np.mean(eq_energy)

# 2. Calculate the standard deviation (how wide the waves bounce)
std_E = np.std(eq_energy)

# 3. Define the upper and lower boundaries of your window
lower_bound = mean_E - std_E
upper_bound = mean_E + std_E

print(f"Your system's average energy is: {mean_E:.2f}")
print(f"The normal fluctuation size is: ±{std_E:.2f}")
print(f"Your acceptable Energy Window is between: {lower_bound:.2f} and {upper_bound:.2f}")

Your system's average energy is: -169.03
The normal fluctuation size is: ±19.26
Your acceptable Energy Window is between: -188.29 and -149.77


In [11]:
# 1. Center your energy data around 0
normalized_energy = eq_energy - mean_E

# 2. Math trick (correlation) to see how much the data overlaps with itself over time
n = len(eq_energy)
overlap = np.correlate(normalized_energy, normalized_energy, mode='full')[n-1:]
overlap /= (np.var(eq_energy) * np.arange(n, 0, -1))

# 3. Find the first step where the correlation drops below 0.368 (which is 1/e)
tau = np.where(overlap < 0.368)[0][0]

# 4. Standard safety practice: double tau to guarantee complete decorrelation
safe_stride = 2 * tau

print(f"The system takes {tau} steps to 'forget' its previous layout.")
print(f"To ensure snapshots are uncorrelated, you must separate them by at least: {safe_stride} steps.")

The system takes 2747 steps to 'forget' its previous layout.
To ensure snapshots are uncorrelated, you must separate them by at least: 5494 steps.


In [12]:
# 1. Create an empty list to hold steps that pass both rules
valid_pool = []
last_accepted_step = -np.inf  # Starts at negative infinity so the first step always passes

# 2. Loop through your steady-state steps and energies
for step, energy in zip(eq_steps, eq_energy):
    
    # Rule 1: Is the energy inside our safe window bounds?
    if lower_bound <= energy <= upper_bound:
        
        # Rule 2: Has enough time passed since our last selection?
        if (step - last_accepted_step) >= safe_stride:
            
            # If both pass, add it to our pool!
            valid_pool.append(step)
            last_accepted_step = step

print(f"Filtering complete!")
print(f"Out of your entire run, we found {len(valid_pool)} perfectly stable, uncorrelated steps.")

Filtering complete!
Out of your entire run, we found 15 perfectly stable, uncorrelated steps.
